# Effect of the plate-position correction on the feature space

`1.fit_position_correction.ipynb` estimates a plate-position effect (the "tilt") in the single-cell features of the validation plates, and `../4b.apply_position_correction.ipynb` subtracts it from every cell.
The tilt is an additive row plus column effect of the well position, estimated for each feature separately: every feature has its own row and column effects, and each plate has one amplitude that scales all of them.
The correction subtracts the amplitude of the plate times the tilt of the well from every cell, so all cells of a well shift by the same vector.

This notebook computes what we need to show the effect of the correction on the features and on the well profiles, and saves it as tables.
`4.plot_effect_on_feature_space.ipynb` draws the figures:

1. The size of the tilt for each feature family (compartment, measurement type, and channel).
2. The wells in the two principal components that matter most for plate row, before and after the correction, and the displacement of each well.
3. The principal component that depends most on the plate row, drawn on the plate grid before and after the correction.

**Inputs** (in this folder, from `1.fit_position_correction.ipynb`)

- `position_correction_fit.npz`: the tilt map and the amplitude of each plate.
- `well_medians.parquet`: the median profile of every well of every plate before the correction. The fit notebook caches it, and git ignores it.

The correction shifts all cells of a well by the same vector, so the median profile of a well after the correction is its median before the correction minus that vector.
We therefore compute the corrected wells from the well medians, without reading single cells.

**Outputs** (in `results/`)

- `tilt_size_by_feature.csv`: the size of the tilt and the family of every feature.
- `well_pca_scores.csv`: the principal component scores of every well before and after the correction.
- `pc_variance_explained_by_row.csv`: the fraction of each component that plate row explains, before and after.

## Import libraries

In [1]:
import pathlib
import sys

import pandas as pd

sys.path.append("../../../utils")
import position_correction_effect_utils as pce
import position_correction_utils as pcu
import validation_position_utils as vpu

## Set paths and parameters

In [2]:
output_dir = pathlib.Path(".")
fit_path = pathlib.Path("position_correction_fit.npz")
well_medians_path = pathlib.Path("well_medians.parquet")
for path in (fit_path, well_medians_path):
    if not path.exists():
        raise FileNotFoundError(
            f"Input does not exist: {path}. Run 1.fit_position_correction.ipynb first."
        )

# folder for the tables
results_dir = pathlib.Path("results")
results_dir.mkdir(exist_ok=True)

# number of principal components of the uncorrected wells
n_components = 10
# seed of the PCA, so the components are reproducible
random_state = 0

## Load the fit and the well medians

Each row of the well medians is one well of one plate, with its treatment and cell type.

In [3]:
fit = pcu.load_fit(fit_path)
medians = pd.read_parquet(well_medians_path)

missing = set(fit["features"]) - set(medians.columns)
if missing:
    raise KeyError(f"{len(missing)} features of the fit are not in the well medians")

print("Features in the fit:", len(fit["features"]))
print("Wells:", len(medians), "on", medians["plate"].nunique(), "plates")
print("Amplitude of each plate:", fit["amplitudes"])

Features in the fit: 2453
Wells: 300 on 5 plates
Amplitude of each plate: {1: 1.050907157423678, 2: 0.9206859977549837, 3: 0.9214871525126132, 4: 0.9497407534636622, 5: 0.8214808321766457}


## Size of the tilt by feature family

The size of a feature is the root mean square of its tilt over the 60 positions of the plate (6 rows by 10 columns), with an amplitude of 1.
The features are z-scored on each plate, so the size is in units of the standard deviation of a single cell.
The families use the channels of the validation plates (`Golgi` is the d2 channel, `PM` in the screen).

In [4]:
sizes = vpu.tilt_size_by_feature(fit)
sizes.sort_values("tilt_rms", ascending=False).to_csv(
    results_dir / "tilt_size_by_feature.csv", index=False
)
sizes.sort_values("tilt_rms", ascending=False).head(10)

,feature,tilt_rms,compartment,measurement,channel
614,Cytoplasm_Texture_DifferenceEntropy_Actin_3_01...,0.690272,Cytoplasm,Texture,Actin
1434,Cells_Texture_DifferenceEntropy_Actin_3_01_256,0.688022,Cells,Texture,Actin
635,Cytoplasm_Texture_DifferenceVariance_Actin_3_0...,0.682990,Cytoplasm,Texture,Actin
655,Cytoplasm_Texture_Entropy_Actin_3_02_256,0.682788,Cytoplasm,Texture,Actin
1455,Cells_Texture_DifferenceVariance_Actin_3_02_256,0.682506,Cells,Texture,Actin
616,Cytoplasm_Texture_DifferenceEntropy_Actin_3_03...,0.681710,Cytoplasm,Texture,Actin
1475,Cells_Texture_Entropy_Actin_3_02_256,0.680022,Cells,Texture,Actin
654,Cytoplasm_Texture_Entropy_Actin_3_01_256,0.679921,Cytoplasm,Texture,Actin
615,Cytoplasm_Texture_DifferenceEntropy_Actin_3_02...,0.679577,Cytoplasm,Texture,Actin
1436,Cells_Texture_DifferenceEntropy_Actin_3_03_256,0.678897,Cells,Texture,Actin


## Wells in principal components, before and after the correction

We fit the principal components on the uncorrected wells, and project both the uncorrected and the corrected wells into them.
This puts the two versions in the same space, so we can compare the wells before and after the correction.
The wells include the compound wells and the DMSO controls.

In [5]:
features = fit["features"]
corrected = pce.correct_well_medians(medians, fit)

pca = pce.fit_pca(medians[features].to_numpy(dtype=float), n_components, random_state)
before = pca.transform(medians[features].to_numpy(dtype=float))
after = pca.transform(corrected[features].to_numpy(dtype=float))

rows, cols = pcu.well_index(medians["well"])
variance_by_row = pd.DataFrame(
    {
        "component": [f"PC{i + 1}" for i in range(n_components)],
        "variance_explained": pca.explained_variance_ratio_,
        "row_explains_before": [
            pce.variance_explained_by(before[:, i], rows) for i in range(n_components)
        ],
        "row_explains_after": [
            pce.variance_explained_by(after[:, i], rows) for i in range(n_components)
        ],
        "col_explains_before": [
            pce.variance_explained_by(before[:, i], cols) for i in range(n_components)
        ],
        "col_explains_after": [
            pce.variance_explained_by(after[:, i], cols) for i in range(n_components)
        ],
    }
)
variance_by_row.to_csv(results_dir / "pc_variance_explained_by_row.csv", index=False)
vpu.well_score_table(medians, before, after).to_csv(
    results_dir / "well_pca_scores.csv", index=False
)

top_components = variance_by_row.nlargest(2, "row_explains_before")["component"]
print("Components that depend most on plate row:", sorted(top_components))
variance_by_row.round(3)

Components that depend most on plate row: ['PC1', 'PC8']


,component,variance_explained,row_explains_before,row_explains_after,col_explains_before,col_explains_after
0,PC1,0.479,0.532,0.004,0.200,0.407
1,PC2,0.152,0.060,0.007,0.271,0.274
2,PC3,0.077,0.078,0.014,0.138,0.087
3,PC4,0.050,0.077,0.011,0.051,0.031
4,PC5,0.045,0.123,0.033,0.042,0.017
5,PC6,0.025,0.025,0.009,0.169,0.093
6,PC7,0.023,0.105,0.007,0.096,0.087
7,PC8,0.017,0.135,0.009,0.170,0.182
8,PC9,0.014,0.003,0.009,0.184,0.162
9,PC10,0.011,0.027,0.005,0.080,0.072
